# 02 Variable Curation Prevent Pce R

Variable curation notebook (R)

Purpose: curate PREVENT / PCE covariates and export intermediate analytic artifacts.

Notes: this notebook is Truveta-specific and may require environment-specific objects, paths, and permissions.

# Initialize Truveta SDK

In [ ]:
library(readr, warn.conflicts = FALSE)
library(arrow, warn.conflicts = FALSE)
library(magrittr, warn.conflicts = FALSE)
library(stringr, warn.conflicts = FALSE)
library(dplyr, warn.conflicts = FALSE)
library(rlang, warn.conflicts = FALSE)
library(data.table, warn.conflicts = FALSE)
library(lubridate, warn.conflicts = FALSE)
library(tidyr, warn.conflicts = FALSE)
library(truveta.notebook.study)

In [ ]:
# Use only one statement below and comment out whichever you are not using.
# con <- create_connection(output_mode = "sparklyr")
con <- create_connection(output_mode = "sparkr")

In [ ]:
study <- get_study(con)
# Use only one statement below and comment out whichever you are not using.
# population <- get_population(con, study, title='PREVENT and PCE')
population <- get_population(con, id='<TRUVETA_POPULATION_ID>')

population

In [ ]:
# Get latest completed active snapshot.
snapshot = get_latest_snapshot(con, population)
snapshot

In [ ]:
# Show tables in the snapshot.
get_tables(con, snapshot)

In [ ]:
%%sql
select * from SearchResult_diabetes
limit 5;

In [ ]:
%%sql
select PersonId, 
    CAST(
        CASE 
            WHEN OnsetDateTime IS NOT NULL THEN from_unixtime(OnsetDateTime) 
            ELSE from_unixtime(RecordedDateTime) 
        END 
    AS DATE) AS diabetes_dt 
from SearchResult_diabetes group by 1,2;

In [ ]:
%%sql
select * from SearchResult_diabetes_ip
limit 5;

In [ ]:
%%sql
create or replace temp view diabetes as select PersonId, 
date(case when OnsetDateTime is not null then from_unixtime(OnsetDateTime) else from_unixtime(RecordedDateTime) end) as diabetes_dt 
from SearchResult_diabetes group by 1,2;
create or replace temp view diabetes1 as select PersonId, diabetes_dt as diabetes_post, 
lag(diabetes_dt,1) over(partition by PersonId order by diabetes_dt) as diabetes_pre from diabetes;
create or replace temp view diabetes2 as select PersonId, min(diabetes_pre) as diabetes_dt from diabetes1
where datediff(day,diabetes_pre,diabetes_post)<730 group by 1;
create or replace temp view diabetes3 as select PersonId, min(diabetes_dt) as diabetes_dt 
from(select * from diabetes2 union select PersonId, from_unixtime(diabetes_dt) as diabetes_dt from SearchResult_diabetes_ip) z group by 1;

In [ ]:
%%sql
select * from SearchResult_cvd
limit 5;

In [ ]:
%%sql
create or replace temp view cvd as select PersonId, 
date(case when OnsetDateTime is not null then from_unixtime(OnsetDateTime) else from_unixtime(RecordedDateTime) end) as cvd_dt 
from SearchResult_cvd group by 1,2;
create or replace temp view cvd1 as select PersonId, cvd_dt as cvd_post, 
lag(cvd_dt,1) over(partition by PersonId order by cvd_dt) as cvd_pre from cvd;
create or replace temp view cvd2 as select PersonId, min(cvd_pre) as cvd_dt from cvd1
where datediff(day,cvd_pre,cvd_post)<730 group by 1;
create or replace temp view cvd3 as select PersonId, min(cvd_dt) as cvd_dt 
from(select * from cvd2 union select PersonId, from_unixtime(cvd_dt) as cvd_dt from SearchResult_cvd_ip) z group by 1;

In [ ]:
%%sql
create or replace temp view cursmk as select PersonId, 
min(date(case when OnsetDateTime is not null then from_unixtime(OnsetDateTime) else from_unixtime(RecordedDateTime) end)) as cursmk_dt 
from SearchResult_cursmk group by 1;

In [ ]:
%%sql
create or replace temp view pcr as select PersonId, from_unixtime(pcr_dtt) as pcr_dt, avg(pcr) as pcr
from SearchResult_pcr_op group by 1,2;
create or replace temp view acr as select PersonId, from_unixtime(acr_dtt) as acr_dt, avg(acr) as acr
from SearchResult_acr_op group by 1,2;

In [ ]:
%%sql
create or replace temp view scre as select PersonId, from_unixtime(scre_dtt) as scre_dt,avg(scre) as scre 
from SearchResult_scre_op 
where scre_unit = 1190110 group by 1,2;
create or replace temp view chol as select PersonId, from_unixtime(chol_dtt) as chol_dt, avg(chol) as chol 
from SearchResult_chol_op 
where chol_unit = 1190110 group by 1,2;
create or replace temp view hdlc as select PersonId, from_unixtime(hdlc_dtt) as hdlc_dt, avg(hdlc) as hdlc 
from SearchResult_hdlc_op 
where hdlc_unit = 1190110 group by 1,2;
create or replace temp view hba1c as select PersonId, from_unixtime(hba1c_dtt) as hba1c_dt, avg(hba1c) as hba1c
from SearchResult_hba1c_op 
group by 1,2;
create or replace temp view bmi as select PersonId, from_unixtime(bmi_dtt) as bmi_dt, avg(bmi) as bmi 
from SearchResult_bmi_op group by 1,2;
create or replace temp view sbp as select PersonId, from_unixtime(sbp_dtt) as sbp_dt, avg(sbp) as sbp 
from SearchResult_sbp_op group by 1,2;

In [ ]:
%%sql
select count(distinct Id) from Person;

In [ ]:
%%sql
select count(distinct PersonId) from scre;

create or replace temp view index_tab_temp as SELECT 
    a.PersonId, a.scre as scre, a.scre_dt, b.sbp  as sbp, b.sbp_dt,
    CASE 
        WHEN GREATEST(DATEDIFF(b.sbp_dt, a.scre_dt), DATEDIFF(a.scre_dt, b.sbp_dt)) < 365 
        THEN  LEAST(a.scre_dt,b.sbp_dt)
    END AS index_dtt_temp
FROM 
    scre a 
INNER JOIN 
    sbp b 
ON 
    a.PersonId = b.PersonId
WHERE GREATEST(DATEDIFF(b.sbp_dt, a.scre_dt), DATEDIFF(a.scre_dt, b.sbp_dt)) < 365 ;


create or replace temp view index_tab_temp_1 as SELECT a.*, b.index_dtt
FROM index_tab_temp a
LEFT JOIN 
(SELECT PersonId, MIN(index_dtt_temp) as index_dtt FROM
index_tab_temp
GROUP BY 1) b
ON a.PersonId=b.PersonId;


create or replace temp view index_tab as 
SELECT PersonId, avg(scre) as scre, avg(sbp) as sbp, MIN(index_dtt) as index_dtt
FROM index_tab_temp_1
WHERE GREATEST(DATEDIFF(scre_dt, index_dtt),DATEDIFF(sbp_dt, index_dtt)) between 0 and 365
GROUP BY 1;

select count(distinct PersonId) from index_tab;


In [ ]:
%%sql
select * from hba1c
limit 5;

In [ ]:
%%sql
select count(distinct PersonId) from index_tab;
create or replace temp view chol10 as select distinct PersonId, index_dtt, sbp, scre, chol 
from (select a.PersonId, index_dtt, sbp, scre,
chol, row_number() over(partition by a.PersonId, index_dtt order by chol_dt desc) as row_id
from index_tab a inner join chol b on a.PersonId = b.PersonId where datediff(day,chol_dt,index_dtt) between 0 and 365*3) z 
where row_id = 1;
select count(distinct PersonId) from chol10;
create or replace temp view hdlc10 as select distinct PersonId, index_dtt, sbp, scre, chol, hdlc 
from (select a.PersonId, index_dtt, sbp, 
scre, chol, hdlc, row_number() over(partition by a.PersonId, index_dtt order by hdlc_dt desc) as row_id
from hdlc a inner join chol10 b on a.PersonId = b.PersonId where datediff(day,hdlc_dt,index_dtt) between 0 and 365*3) z 
where row_id = 1;
select count(distinct PersonId) from hdlc10;
create or replace temp view bmi10 as select distinct PersonId, index_dtt, sbp, scre, chol, hdlc, bmi 
from (select a.PersonId, index_dtt, 
sbp, scre, chol, hdlc, bmi, row_number() over(partition by a.PersonId, index_dtt order by bmi_dt desc) as row_id
from bmi a inner join hdlc10 b on a.PersonId = b.PersonId where datediff(day,bmi_dt,index_dtt) between 0 and 365*3) z 
where row_id = 1;
select count(distinct PersonId) from bmi10;

In [ ]:
%%sql
create or replace temp view hba1c10 as select distinct PersonId, index_dtt, sbp, scre, chol, hdlc, bmi,hba1c 
from (select a.PersonId, index_dtt, 
sbp, scre, chol, hdlc, bmi, hba1c, row_number() over(partition by a.PersonId, index_dtt order by hba1c_dt desc) as row_id
from bmi10 a left join hba1c b on a.PersonId = b.PersonId where datediff(day,hba1c_dt,index_dtt) between 0 and 365*3) z 
where row_id = 1;
select count(distinct PersonId) from hba1c10;


create or replace temp view acr10 as select distinct PersonId, index_dtt,sbp,scre, chol, hdlc, bmi, hba1c, acr
from (select a.PersonId, index_dtt, 
sbp, scre, chol, hdlc, bmi, hba1c, b.acr, row_number() over(partition by a.PersonId, index_dtt order by acr_dt desc) as row_id
from hba1c10 a left join acr b on a.PersonId = b.PersonId) z 
where row_id = 1;
select count(distinct PersonId) from acr10;

create or replace temp view dat00 as select distinct PersonId, index_dtt, sbp, scre, chol, hdlc, bmi, hba1c, acr 
from (select a.PersonId, index_dtt, 
sbp, scre, chol, hdlc, bmi, hba1c, acr, row_number() over(partition by a.PersonId order by index_dtt) as row_id
from acr10 a left join cvd3 b on a.PersonId = b.PersonId where cvd_dt>index_dtt or cvd_dt is null) z 
where row_id = 1;
select count(distinct PersonId) from dat00;


create or replace temp view antihtn0 as select distinct a.PersonId
from SearchResult_antihtn a inner join dat00 b on a.PersonId = b.PersonId
where index_dtt between from_unixtime(StartDateTime) and from_unixtime(StopDateTime) 
or datediff(day,from_unixtime(AuthoredOnDateTime),index_dtt) between 0 and 180;
create or replace temp view statin0 as select distinct a.PersonId
from SearchResult_statin a inner join dat00 b on a.PersonId = b.PersonId
where index_dtt between from_unixtime(StartDateTime) and from_unixtime(StopDateTime) 
or datediff(day,from_unixtime(AuthoredOnDateTime),index_dtt) between 0 and 180;
select count(distinct PersonId) from chol10;
select count(distinct PersonId) from hdlc10;
select count(distinct PersonId) from bmi10;
select count(distinct PersonId),count(*) from dat00;

In [ ]:
%%sql
create or replace temp view dat10_temp as select distinct a.PersonId, index_dtt, datediff(day,from_unixtime(BirthDateTime),index_dtt)/365.25 AS age,
(CASE WHEN GenderConceptId = 1065405 THEN 1 WHEN GenderConceptId = 1065406 THEN 0 END) AS female,scre, sbp, bmi, chol, hdlc, hba1c, acr,
(case when diabetes_dt<=index_dtt then 1 else 0 end) as diabetes, (case when cursmk_dt<=index_dtt then 1 else 0 end) as cursmk,
(case when m.PersonId is not null then 1 else 0 end) as antihtn, (case when n.PersonId is not null then 1 else 0 end) as statin,
(case when endt_mi is not null then 1 else 0 end) as event_mi, (case when endt_str is not null then 1 else 0 end) as event_str,
datediff(day,index_dtt,(case when endt_mi is not null then endt_mi else from_unixtime(end_dtt) end))/365.25 as fu_mi,
datediff(day,index_dtt,(case when endt_str is not null then endt_str else from_unixtime(end_dtt) end))/365.25 as fu_str,
datediff(day,index_dtt,(case when endt_hf is not null then endt_hf else from_unixtime(end_dtt) end))/365.25 as fu_hf,
(case when endt_hf is not null then 1 else 0 end) as event_hf, (case when IsExpired then 1 else 0 end) AS event_death
from dat00 a left join diabetes3 b on a.PersonId = b.PersonId left join cursmk c on a.PersonId = c.PersonId 
left join SearchResult_inc_mi d on a.PersonId = d.PersonId left join SearchResult_inc_str e on a.PersonId = e.PersonId 
left join SearchResult_inc_hf f on a.PersonId = f.PersonId left join SearchResult_end g on a.PersonId = g.PersonId 
left join SearchResult_demo h on a.PersonId = h.PersonId  left join antihtn0 m on a.PersonId = m.PersonId
left join statin0 n on a.PersonId = n.PersonId where datediff(day,index_dtt,from_unixtime(end_dtt))>0;

In [ ]:
%%sql
create or replace temp view dat10 as select a.*, 
(case when event_mi=1 or event_str=1 then 1 else 0 end) as ascvd, 
(case when event_mi=1 or event_str=1 or event_hf=1 then 1 else 0 end) as cvd
from dat10_temp a;

select * from dat10
limit 5;

In [ ]:
%%sql
select count(*), avg(age), avg(female), avg(sbp), avg(antihtn), avg(diabetes), avg(cursmk), avg(chol), avg(hdlc), avg(statin), 
avg(bmi), avg(scre), sum(event_mi), sum(event_str), sum(event_hf), sum(event_death), 
avg(fu_mi), avg(fu_str), avg(fu_hf), avg(ascvd), avg(cvd) from dat10;

select sum(case when index_dtt is null then 1 else 0 end), sum(case when age is null then 1 else 0 end), 
sum(case when female is null then 1 else 0 end), sum(case when sbp is null then 1 else 0 end),
sum (case when antihtn is null then 1 else 0 end), sum(case when diabetes is null then 1 else 0 end), 
sum(case when cursmk is null then 1 else 0 end), sum(case when chol is null then 1 else 0 end), sum(case when hdlc is null then 1 else 0 end),
sum(case when statin is null then 1 else 0 end), sum(case when bmi is null then 1 else 0 end),
sum(case when scre is null then 1 else 0 end), sum(case when event_mi is null then 1 else 0 end), 
sum(case when event_str is null then 1 else 0 end), sum(case when event_hf is null then 1 else 0 end), 
sum(case when event_death is null then 1 else 0 end), 
sum(case when fu_mi is null then 1 else 0 end), sum(case when fu_str is null then 1 else 0 end), sum(case when fu_hf is null then 1 else 0 end), 
sum(case when ascvd is null then 1 else 0 end), sum(case when cvd is null then 1 else 0 end)
from dat10;

In [ ]:
sqldat <- "SELECT * FROM dat10"
dat <- load_sql_table(con, snapshot, sqldat, view_name='tbl_dat_new_index', output_mode = 'sparklyr')
path = file.path('data/PREVENT_PCE_COHORT_12_10_2024.parquet')
save_artifacts_data(con,study,dat,path)

Saved_df = load_artifacts_data(con,study,path)
display(Saved_df)